# Lab Assignment Four: Multi-Layer Perceptron

**Team Members:**
| Role | Name |
|---|---|
| Teammate 1 — Data Pipeline & Setup | |
| Teammate 2 — Preprocessing & 3-Layer MLP | |
| Teammate 3 — Deep Networks & Adaptive Learning | |

**Dataset:** ACS 2017 US Census Tract Data  
**Task:** Predict child poverty rate (4-class classification)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

---
## 1. Load, Split & Balance
*(Teammate 1)*

### 1.1 Load & Clean

In [ ]:
df = pd.read_csv('acs2017_census_tract_data.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df = df.dropna()
print('Shape after dropping NaN:', df.shape)

In [ ]:
string_cols = df.select_dtypes(include=['object']).columns.tolist()
print('String columns:', string_cols)

le = LabelEncoder()
for col in string_cols:
    df[col] = le.fit_transform(df[col])

df.head()

### 1.2 County Variable Decision

**Discussion:** *(Should the `County` variable be kept or removed? Explain your reasoning.)*

In [ ]:
# Uncomment one option:
# df = df.drop(columns=['County'])  # remove County
# pass                               # keep County

print('Columns:', df.columns.tolist())

### 1.3 Quantize Target & Balance Dataset

In [ ]:
df['ChildPovertyClass'] = pd.qcut(df['ChildPoverty'], q=4, labels=[0, 1, 2, 3]).astype(int)

print('Class distribution:')
print(df['ChildPovertyClass'].value_counts().sort_index())

df['ChildPovertyClass'].value_counts().sort_index().plot(kind='bar', title='Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

**Discussion:** *(Should balancing be applied to both the training set and the test set? Explain.)*

### 1.4 Train/Test Split (80/20)

In [ ]:
X = df.drop(columns=['ChildPoverty', 'ChildPovertyClass']).values
y = df['ChildPovertyClass'].values
all_cols = df.drop(columns=['ChildPoverty', 'ChildPovertyClass']).columns.tolist()
categorical_col_names = string_cols.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Train class counts:', np.bincount(y_train))
print('Test class counts: ', np.bincount(y_test))

In [ ]:
def one_hot_target(y, num_classes=4):
    Y = np.zeros((len(y), num_classes))
    Y[np.arange(len(y)), y] = 1
    return Y

Y_train = one_hot_target(y_train)
Y_test  = one_hot_target(y_test)
print('Y_train shape:', Y_train.shape)

---
## 2. MLP Helper Functions & Base Classes
*(Shared — build together before splitting)*

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(a):
    return a * (1 - a)

def softmax(z):
    e = np.exp(z - np.max(z, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def glorot(fan_in, fan_out):
    limit = np.sqrt(6 / (fan_in + fan_out))
    return np.random.uniform(-limit, limit, (fan_in, fan_out))

def cross_entropy(Y_hat, Y):
    return -np.sum(Y * np.log(Y_hat + 1e-15)) / Y.shape[0]

def accuracy(Y_hat, Y):
    return np.mean(np.argmax(Y_hat, axis=1) == np.argmax(Y, axis=1))

In [ ]:
class MLP2Layer:
    """Two-layer MLP with mini-batch SGD, cross-entropy loss, Glorot init, sigmoid activations."""

    def __init__(self, input_size, hidden_size, output_size, lr=0.01, batch_size=64, epochs=100, seed=42):
        np.random.seed(seed)
        self.lr, self.batch_size, self.epochs = lr, batch_size, epochs
        self.W1 = glorot(input_size, hidden_size);  self.b1 = np.zeros((1, hidden_size))
        self.W2 = glorot(hidden_size, output_size); self.b2 = np.zeros((1, output_size))
        self.loss_history = []

    def forward(self, X):
        self.A1 = sigmoid(X @ self.W1 + self.b1)
        self.A2 = softmax(self.A1 @ self.W2 + self.b2)
        return self.A2

    def backward(self, X, Y):
        n = X.shape[0]
        dZ2 = self.A2 - Y
        dW2 = self.A1.T @ dZ2 / n;  db2 = dZ2.mean(0, keepdims=True)
        dZ1 = (dZ2 @ self.W2.T) * sigmoid_deriv(self.A1)
        dW1 = X.T @ dZ1 / n;        db1 = dZ1.mean(0, keepdims=True)
        self.W2 -= self.lr * dW2;  self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1;  self.b1 -= self.lr * db1

    def fit(self, X, Y):
        self.loss_history = []
        for epoch in range(self.epochs):
            idx = np.random.permutation(len(X))
            Xs, Ys = X[idx], Y[idx]
            for s in range(0, len(X), self.batch_size):
                self.forward(Xs[s:s+self.batch_size])
                self.backward(Xs[s:s+self.batch_size], Ys[s:s+self.batch_size])
            loss = cross_entropy(self.forward(X), Y)
            self.loss_history.append(loss)
            if (epoch + 1) % 10 == 0:
                print(f'Epoch {epoch+1}/{self.epochs}  loss={loss:.4f}')

    def predict(self, X):
        return self.forward(X)

In [ ]:
class MLPNLayer:
    """N-layer MLP. Tracks gradient magnitudes per layer per epoch."""

    def __init__(self, layer_sizes, lr=0.01, batch_size=64, epochs=100, seed=42):
        np.random.seed(seed)
        self.lr, self.batch_size, self.epochs = lr, batch_size, epochs
        self.n = len(layer_sizes) - 1
        self.W = [glorot(layer_sizes[i], layer_sizes[i+1]) for i in range(self.n)]
        self.b = [np.zeros((1, layer_sizes[i+1])) for i in range(self.n)]
        self.loss_history = []
        self.grad_history  = [[] for _ in range(self.n)]

    def forward(self, X):
        self.A = [X]
        for i in range(self.n - 1):
            self.A.append(sigmoid(self.A[-1] @ self.W[i] + self.b[i]))
        self.A.append(softmax(self.A[-1] @ self.W[-1] + self.b[-1]))
        return self.A[-1]

    def backward(self, X, Y):
        n = X.shape[0]
        dW = [None] * self.n;  db = [None] * self.n
        delta = self.A[-1] - Y
        dW[-1] = self.A[-2].T @ delta / n
        db[-1] = delta.mean(0, keepdims=True)
        for i in range(self.n - 2, -1, -1):
            delta = (delta @ self.W[i+1].T) * sigmoid_deriv(self.A[i+1])
            dW[i] = self.A[i].T @ delta / n
            db[i] = delta.mean(0, keepdims=True)
        self._grad_mags = [np.mean(np.abs(dW[i])) for i in range(self.n)]
        for i in range(self.n):
            self.W[i] -= self.lr * dW[i]
            self.b[i] -= self.lr * db[i]

    def fit(self, X, Y):
        self.loss_history = []
        self.grad_history  = [[] for _ in range(self.n)]
        for epoch in range(self.epochs):
            idx = np.random.permutation(len(X))
            Xs, Ys = X[idx], Y[idx]
            epoch_mags = np.zeros(self.n);  nb = 0
            for s in range(0, len(X), self.batch_size):
                self.forward(Xs[s:s+self.batch_size])
                self.backward(Xs[s:s+self.batch_size], Ys[s:s+self.batch_size])
                epoch_mags += self._grad_mags;  nb += 1
            for i in range(self.n):
                self.grad_history[i].append(epoch_mags[i] / nb)
            loss = cross_entropy(self.forward(X), Y)
            self.loss_history.append(loss)
            if (epoch + 1) % 10 == 0:
                print(f'Epoch {epoch+1}/{self.epochs}  loss={loss:.4f}')

    def predict(self, X):
        return self.forward(X)

    def plot_grad_magnitudes(self, title):
        plt.figure(figsize=(9, 4))
        for i, mags in enumerate(self.grad_history):
            plt.plot(mags, label=f'Layer {i+1}')
        plt.title(title)
        plt.xlabel('Epoch');  plt.ylabel('Avg Gradient Magnitude')
        plt.legend();  plt.tight_layout();  plt.show()

---
## 3. Pre-processing & Initial Modeling
*(Teammate 1: Model 1 — Teammate 2: Models 2 & 3)*

### Model 1 — No Normalization, No One-Hot Encoding
*(Teammate 1)*

In [ ]:
hidden_size = 64
output_size = 4

mlp1 = MLP2Layer(X_train.shape[1], hidden_size, output_size, lr=0.01, epochs=100)
mlp1.fit(X_train, Y_train)

acc1_train = accuracy(mlp1.predict(X_train), Y_train)
acc1_test  = accuracy(mlp1.predict(X_test),  Y_test)
print(f'Model 1  |  Train: {acc1_train:.4f}  Test: {acc1_test:.4f}')

In [ ]:
plt.plot(mlp1.loss_history)
plt.title('Model 1 — Loss vs Epochs (No Preprocessing)')
plt.xlabel('Epoch');  plt.ylabel('Cross-Entropy Loss')
plt.tight_layout();  plt.show()

### Model 2 — Normalize Continuous Features
*(Teammate 2)*

In [ ]:
cont_indices = [i for i, col in enumerate(all_cols) if col not in categorical_col_names]
print(f'{len(cont_indices)} continuous features, {len(all_cols) - len(cont_indices)} categorical')

scaler = StandardScaler()
X_train_norm = X_train.copy().astype(float)
X_test_norm  = X_test.copy().astype(float)
X_train_norm[:, cont_indices] = scaler.fit_transform(X_train[:, cont_indices])
X_test_norm[:,  cont_indices] = scaler.transform(X_test[:, cont_indices])

In [ ]:
mlp2 = MLP2Layer(X_train_norm.shape[1], hidden_size, output_size, lr=0.01, epochs=100)
mlp2.fit(X_train_norm, Y_train)

acc2_train = accuracy(mlp2.predict(X_train_norm), Y_train)
acc2_test  = accuracy(mlp2.predict(X_test_norm),  Y_test)
print(f'Model 2  |  Train: {acc2_train:.4f}  Test: {acc2_test:.4f}')

In [ ]:
plt.plot(mlp2.loss_history)
plt.title('Model 2 — Loss vs Epochs (Normalized)')
plt.xlabel('Epoch');  plt.ylabel('Cross-Entropy Loss')
plt.tight_layout();  plt.show()

### Model 3 — Normalize Continuous + One-Hot Encode Categorical
*(Teammate 2)*

In [ ]:
cat_indices = [i for i, col in enumerate(all_cols) if col in categorical_col_names]
all_X = np.vstack([X_train, X_test])

def build_ohe(X_norm, cat_idx, all_X):
    cont_idx = [i for i in range(X_norm.shape[1]) if i not in cat_idx]
    parts = [X_norm[:, cont_idx]]
    for i in cat_idx:
        n_cats = int(all_X[:, i].max()) + 1
        parts.append(np.eye(n_cats)[X_norm[:, i].astype(int)])
    return np.hstack(parts)

X_train_ohe = build_ohe(X_train_norm, cat_indices, all_X)
X_test_ohe  = build_ohe(X_test_norm,  cat_indices, all_X)
print('X_train_ohe shape:', X_train_ohe.shape)

In [ ]:
mlp3 = MLP2Layer(X_train_ohe.shape[1], hidden_size, output_size, lr=0.01, epochs=100)
mlp3.fit(X_train_ohe, Y_train)

acc3_train = accuracy(mlp3.predict(X_train_ohe), Y_train)
acc3_test  = accuracy(mlp3.predict(X_test_ohe),  Y_test)
print(f'Model 3  |  Train: {acc3_train:.4f}  Test: {acc3_test:.4f}')

In [ ]:
plt.plot(mlp3.loss_history)
plt.title('Model 3 — Loss vs Epochs (Normalized + One-Hot)')
plt.xlabel('Epoch');  plt.ylabel('Cross-Entropy Loss')
plt.tight_layout();  plt.show()

### Comparison: Models 1–3
*(Teammate 2)*

In [ ]:
results = pd.DataFrame({
    'Model':          ['1 — No Preprocessing', '2 — Normalized', '3 — Normalized + OHE'],
    'Train Accuracy': [acc1_train, acc2_train, acc3_train],
    'Test Accuracy':  [acc1_test,  acc2_test,  acc3_test]
})
print(results.to_string(index=False))

plt.figure(figsize=(10, 4))
plt.plot(mlp1.loss_history, label='Model 1')
plt.plot(mlp2.loss_history, label='Model 2')
plt.plot(mlp3.loss_history, label='Model 3')
plt.title('Loss vs Epochs — Models 1, 2, 3')
plt.xlabel('Epoch');  plt.ylabel('Cross-Entropy Loss')
plt.legend();  plt.tight_layout();  plt.show()

**Discussion:** *(Are there meaningful performance differences across models 1–3? Explain why normalization and one-hot encoding help or don't help for this dataset.)*

> **Note:** For all remaining models, use `X_train_ohe` / `X_test_ohe` (normalized + one-hot).

---
## 4. Deeper MLPs with Gradient Magnitude Tracking
*(Teammate 2: 3-layer — Teammate 3: 4- and 5-layer)*

In [ ]:
input_size = X_train_ohe.shape[1]

### 3-Layer MLP
*(Teammate 2)*

In [ ]:
mlp_3L = MLPNLayer([input_size, 64, 32, output_size], lr=0.01, epochs=100)
mlp_3L.fit(X_train_ohe, Y_train)

acc_3L = accuracy(mlp_3L.predict(X_test_ohe), Y_test)
print(f'3-Layer  |  Test Accuracy: {acc_3L:.4f}')

In [ ]:
plt.plot(mlp_3L.loss_history)
plt.title('3-Layer MLP — Loss vs Epochs')
plt.xlabel('Epoch');  plt.ylabel('Cross-Entropy Loss')
plt.tight_layout();  plt.show()

mlp_3L.plot_grad_magnitudes('3-Layer MLP — Gradient Magnitudes per Layer')

### 4-Layer MLP
*(Teammate 3)*

In [ ]:
mlp_4L = MLPNLayer([input_size, 64, 64, 32, output_size], lr=0.01, epochs=100)
mlp_4L.fit(X_train_ohe, Y_train)

acc_4L = accuracy(mlp_4L.predict(X_test_ohe), Y_test)
print(f'4-Layer  |  Test Accuracy: {acc_4L:.4f}')

In [ ]:
plt.plot(mlp_4L.loss_history)
plt.title('4-Layer MLP — Loss vs Epochs')
plt.xlabel('Epoch');  plt.ylabel('Cross-Entropy Loss')
plt.tight_layout();  plt.show()

mlp_4L.plot_grad_magnitudes('4-Layer MLP — Gradient Magnitudes per Layer')

### 5-Layer MLP
*(Teammate 3)*

In [ ]:
mlp_5L = MLPNLayer([input_size, 64, 64, 32, 32, output_size], lr=0.01, epochs=100)
mlp_5L.fit(X_train_ohe, Y_train)

acc_5L = accuracy(mlp_5L.predict(X_test_ohe), Y_test)
print(f'5-Layer  |  Test Accuracy: {acc_5L:.4f}')

In [ ]:
plt.plot(mlp_5L.loss_history)
plt.title('5-Layer MLP — Loss vs Epochs')
plt.xlabel('Epoch');  plt.ylabel('Cross-Entropy Loss')
plt.tight_layout();  plt.show()

mlp_5L.plot_grad_magnitudes('5-Layer MLP — Gradient Magnitudes per Layer')

---
## 5. Adaptive Learning — RMSProp on 5-Layer Network
*(Teammate 3)*

In [ ]:
class MLPRMSProp(MLPNLayer):
    """5-layer MLP with RMSProp: v = rho*v + (1-rho)*g^2,  W -= lr/sqrt(v+eps)*g"""

    def __init__(self, layer_sizes, lr=0.001, batch_size=64, epochs=100, rho=0.9, eps=1e-8, seed=42):
        super().__init__(layer_sizes, lr, batch_size, epochs, seed)
        self.rho, self.eps = rho, eps
        self.vW = [np.zeros_like(w) for w in self.W]
        self.vb = [np.zeros_like(b) for b in self.b]

    def backward(self, X, Y):
        n = X.shape[0]
        dW = [None] * self.n;  db = [None] * self.n
        delta = self.A[-1] - Y
        dW[-1] = self.A[-2].T @ delta / n
        db[-1] = delta.mean(0, keepdims=True)
        for i in range(self.n - 2, -1, -1):
            delta = (delta @ self.W[i+1].T) * sigmoid_deriv(self.A[i+1])
            dW[i] = self.A[i].T @ delta / n
            db[i] = delta.mean(0, keepdims=True)
        self._grad_mags = [np.mean(np.abs(dW[i])) for i in range(self.n)]
        for i in range(self.n):
            self.vW[i] = self.rho * self.vW[i] + (1 - self.rho) * dW[i]**2
            self.vb[i] = self.rho * self.vb[i] + (1 - self.rho) * db[i]**2
            self.W[i] -= self.lr / np.sqrt(self.vW[i] + self.eps) * dW[i]
            self.b[i] -= self.lr / np.sqrt(self.vb[i] + self.eps) * db[i]

In [ ]:
mlp_5L_rms = MLPRMSProp([input_size, 64, 64, 32, 32, output_size], lr=0.001, epochs=100, rho=0.9)
mlp_5L_rms.fit(X_train_ohe, Y_train)

acc_5L_rms = accuracy(mlp_5L_rms.predict(X_test_ohe), Y_test)
print(f'5-Layer RMSProp  |  Test Accuracy: {acc_5L_rms:.4f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(mlp_5L.loss_history,     label='5-Layer SGD')
plt.plot(mlp_5L_rms.loss_history, label='5-Layer RMSProp')
plt.title('5-Layer: SGD vs RMSProp — Loss vs Epochs')
plt.xlabel('Epoch');  plt.ylabel('Cross-Entropy Loss')
plt.legend();  plt.tight_layout();  plt.show()

mlp_5L_rms.plot_grad_magnitudes('5-Layer RMSProp — Gradient Magnitudes per Layer')

**Discussion:** *(Which adaptive method did you choose and why? Compare the 5-layer SGD vs RMSProp in terms of convergence speed, final accuracy, and gradient magnitudes.)*

---
## 6. Exceptional Work
*(5000-level: free choice)*

In [ ]:
# Ideas: confusion matrix, per-class accuracy, hyperparameter sweep,
# train vs test accuracy curves, AdaM implementation (required for 7000-level)


---
## Summary

| Model | Test Accuracy |
|---|---|
| 1 — 2-layer, no preprocessing | |
| 2 — 2-layer, normalized | |
| 3 — 2-layer, normalized + OHE | |
| 3-layer MLP | |
| 4-layer MLP | |
| 5-layer MLP (SGD) | |
| 5-layer MLP (RMSProp) | |